# Gold - Fato de Hospedagens

Esta tabela representa a camada analítica detalhada das despesas de hospedagem.

A fato especializada preserva atributos específicos como cidade, período da hospedagem e quantidade de noites, que não pertencem à fato consolidada de viagens.

In [0]:
from pyspark.sql import functions as F

df_hospedagens_silver = spark.table(
    "viagens_teste.silver.hospedagens"
)

print(
    "Registros Silver:",
    df_hospedagens_silver.count()
)

### Construção da fato de hospedagens

Selecionamos os atributos necessários para análises financeiras, temporais e operacionais de hospedagem.

In [0]:
df_fato_hospedagens = (
    df_hospedagens_silver
    .select(
        F.col("id_hospedagem"),

        F.col("viajante"),

        F.col("data_checkin"),
        F.col("data_checkout"),

        F.col("centro_custo"),
        F.col("diretoria"),

        F.col("cidade"),

        F.col("valor_numerico").alias("valor"),

        F.col("qtd_noites"),

        F.col("situacao"),
        F.col("agencia"),

        F.col("trimestre"),

        F.lit("Hospedagem").alias("tipo_despesa")
    )
)

In [0]:
display(df_fato_hospedagens)

### Validação da Gold

Validamos a quantidade de registros e os principais campos antes de persistir a tabela.

In [0]:
print(
    "Total de hospedagens:",
    df_fato_hospedagens.count()
)

print(
    "Cidades ausentes:",
    df_fato_hospedagens
    .filter(F.col("cidade").isNull())
    .count()
)

print(
    "Quantidade de noites ausente:",
    df_fato_hospedagens
    .filter(F.col("qtd_noites").isNull())
    .count()
)

In [0]:
(
    df_fato_hospedagens.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "viagens_teste.gold.fato_hospedagens"
    )
)

In [0]:
display(
    spark.table(
        "viagens_teste.gold.fato_hospedagens"
    )
)